In [1]:
import re
import pandas as pd

def read_whatsapp_chat(file_path: str) -> pd.DataFrame:

    encryption_message = "Messages and calls are end-to-end encrypted."
    media_pattern = "<Media omitted>"
    email_pattern = r'[A-Za-z0-9,._%+-]+@[A-Za-z0-9,.-]+\.[A-Za-z]{2,}'
    url_pattern = r'https?://\S+'
    edited_message = "<This message was edited>"
    deleted_message = "You deleted this message"
    null_message = "null"
    created_group_message = "created group"
    added_you_to_group_message = "added you"
    tagging_pattern = r'@[\w]+'

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    filtered_lines = []

    for line in lines:
        if (
            encryption_message not in line and 
            deleted_message not in line and 
            null_message != line.split(" ")[-1] and 
            media_pattern not in line and 
            created_group_message not in line and 
            added_you_to_group_message not in line and 
            not re.search(email_pattern, line) and 
            not re.search(url_pattern, line)
        ):
            line = line.replace(edited_message, "").strip()
            line = re.sub(tagging_pattern, "", line).strip()
            filtered_lines.append(line)

    pattern = r'(\d{2}/\d{2}/\d{2}, \d{1,2}:\d{2}\s?[ap]m) - (.*?): (.*)'
    content = '\n'.join(filtered_lines)

    messages = re.findall(pattern, content, re.DOTALL)

    df = pd.DataFrame(messages, columns=['timestamp', 'sender', 'message'])

    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%m/%y, %I:%M %p')
    return df


In [2]:
from pathlib import Path

all_chats = {}
data_dictionary = Path("../data/private")
for file in data_dictionary.glob('*.txt'):
    file_name = file.stem
    all_chats[file_name] = read_whatsapp_chat(file)

In [3]:
text_sequence = ""
for file_name in all_chats.keys():
    text_sequence += " ".join(all_chats[file_name]['message'].values)

len(text_sequence)

335889

In [4]:
with open("../output/combined_text.txt","w", encoding = "utf-8") as f:
    f.write(text_sequence)